# GIK-IceChain — Run Colab : C1 (ingestion) + C2 (exceedance)

Exécute **les deux premières étapes du pipeline sur Colab**, en écrivant dans le store MinIO :

1. **C1 — `convert`** : ingestion IFS ENS GRIB2 → store IceChunk source (toute la fenêtre, ré-ingestion complète).
2. **C2 — `exceedance`** : accumulations + exceedance GEV adaptative → `exceedance-zarr` sur MinIO.

**C3 (risk) tourne en local** — voir `scripts/run_c3_local.sh` (lit `exceedance-zarr` depuis MinIO).

Fenêtre cible : **2024-09-01 → 2025-01-31** (5 mois, OND-2024 centré), **entièrement ré-ingérée**.

### Identifiants MinIO
Lus dans l'ordre : **Colab Secrets** (`MINIO`, `MINIO_ACCESS_KEY`, `MINIO_SECRET_KEY`), variables d'env, puis `.env`. Sur Colab, ajoutez-les dans l'onglet 🔑 *Secrets*. Pas de fallback synthétique : si le store est injoignable, les cellules lèvent.


## 0. Paramètres — éditez ici

> ⚠️ **Limite de session Colab.** 5 mois ≈ 153 jours ; C1 (re-téléchargement GRIB ~1 Go/jour) **+** C2 dépassent **largement** une session Colab (12 h gratuit / 24 h Pro). **Lancez par sous-blocs** : éditez `START`/`END` (p.ex. 1 semaine), exécutez, puis avancez la fenêtre à la session suivante. C1 est idempotent (`append`) et C2 **saute les dates déjà écrites** — repris proprement bloc par bloc.


In [ ]:
# ----- PARAMS (éditez) -----
# Fenêtre cible globale : 2024-09-01 -> 2025-01-31, ENTIÈREMENT ré-ingérée. Traitez-la par sous-blocs.
START      = "2024-09-01"          # début du sous-bloc courant (incl.)
END        = "2024-09-07"          # fin du sous-bloc courant (incl.) — ~1 semaine pour tenir dans la session
RUN_C1     = True                  # C1 : ré-ingère TOUTE la fenêtre [START, END] (convert idempotent)
RUN_C2     = True                  # C2 exceedance -> écrit exceedance-zarr sur MinIO
C2_WORKERS = 1                     # 1 = séquentiel (RAM ~6 Go, robuste). Augmentez si la VM a + de RAM.
CONFIG     = "configs/default.yaml"
print(f"Sous-bloc {START}..{END} | RUN_C1={RUN_C1} RUN_C2={RUN_C2} workers={C2_WORKERS}")

## 1. Setup — repo, install, creds MinIO, prérequis

In [ ]:
import os, sys, subprocess
from pathlib import Path


def _bootstrap() -> Path:
    for p in (Path.cwd(), *Path.cwd().parents):
        if (p / "configs" / "default.yaml").exists():
            return p
    repo = Path.cwd() / "gik-icechain"
    if not repo.exists():
        subprocess.run(["git", "clone", "--depth", "1",
                        "https://github.com/hashirama21/gik-icechain.git", str(repo)], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", f"{repo}[dev]"], check=True)
    return repo


REPO = _bootstrap()
DATA = REPO / "data"
sys.path.insert(0, str(REPO / "src"))


def _secret(name: str, default: str = "") -> str:
    try:
        from google.colab import userdata
        v = userdata.get(name)
        if v:
            return v
    except Exception:
        pass
    return os.environ.get(name, default)


_env = REPO / ".env"
if _env.exists():
    for _l in _env.read_text().splitlines():
        if _l and not _l.startswith("#") and "=" in _l:
            _k, _v = _l.split("=", 1)
            os.environ.setdefault(_k.strip(), _v.strip())

_minio = _secret("MINIO") or _secret("MINIO_ENDPOINT_URL")
ENDPOINT = _minio if _minio.startswith("http") else (f"http://{_minio}" if _minio else "")
KEY = _secret("MINIO_ACCESS_KEY") or _secret("AWS_ACCESS_KEY_ID")
SECRET = _secret("MINIO_SECRET_KEY") or _secret("AWS_SECRET_ACCESS_KEY")
if not (ENDPOINT and KEY and SECRET):
    raise RuntimeError(
        "Creds MinIO requis (MINIO/MINIO_ACCESS_KEY/MINIO_SECRET_KEY via Colab Secrets, env, ou .env).")

# Les appels CLI tournent en sous-processus et relisent le config (endpoint vide) :
# le endpoint MinIO est résolu via ces variables d'environnement (héritées par env=os.environ).
os.environ["AWS_ENDPOINT_URL"] = ENDPOINT
os.environ["AWS_ACCESS_KEY_ID"] = KEY
os.environ["AWS_SECRET_ACCESS_KEY"] = SECRET
os.environ.setdefault("AWS_REGION", "eu-west-1")
os.environ.setdefault("ECCODES_PYTHON_USE_FINDLIBS", "1")

if not (DATA / "cmorph_thresholds").exists():
    tools = [sys.executable, str(REPO / "scripts" / "tools.py")]
    subprocess.run([*tools, "download", "--component", "all"], check=True)
    subprocess.run([*tools, "download-thresholds"], check=True)

from gik_icechain.shared.config import load_config

cfg = load_config(REPO / CONFIG)
cfg.outputs.endpoint_url = ENDPOINT
STORAGE_OPTIONS = {"endpoint_url": ENDPOINT}
print("Repo            :", REPO)
print("MinIO           :", ENDPOINT)
print("Source store    :", cfg.outputs.icechunk_store_uri)
print("Exceedance store:", cfg.outputs.exceedance_store_uri)

## 2. C1 — ingestion (`convert`) de toute la fenêtre

Ré-ingère **tout** le sous-bloc `[START, END]` dans le store source IceChunk. `convert` est idempotent (`append` via `create_or_open`).

In [ ]:
if RUN_C1:
    cmd = [sys.executable, "-m", "gik_icechain", "convert",
           "--start", START, "--end", END, "--config", str(REPO / CONFIG)]
    print("C1 :", " ".join(cmd), flush=True)
    rc = subprocess.run(cmd, env=os.environ).returncode
    assert rc == 0, f"C1 convert a échoué (exit {rc})"
    print("C1 OK")
else:
    print("C1 ignoré (RUN_C1=False).")

## 3. C2 — exceedance (`exceedance`)

Lit le store source IceChunk, calcule l'exceedance GEV adaptative pour `[START, END]`, écrit `exceedance-zarr` sur MinIO. Le store a été vidé : le premier sous-bloc le recrée (`mode=w`), les suivants font `append` (dates déjà présentes ignorées).

In [ ]:
if RUN_C2:
    cmd = [sys.executable, "-m", "gik_icechain", "exceedance",
           "--store", cfg.outputs.icechunk_store_uri,
           "--output", cfg.outputs.exceedance_store_uri,
           "--start", START, "--end", END,
           "--workers", str(C2_WORKERS),
           "--config", str(REPO / CONFIG)]
    print("C2 :", " ".join(cmd), flush=True)
    rc = subprocess.run(cmd, env=os.environ).returncode
    assert rc == 0, f"C2 exceedance a échoué (exit {rc})"
    print("C2 OK")
else:
    print("C2 ignoré (RUN_C2=False).")

## 4. Vérification — store exceedance & garde-fou anti-corruption

Vérifie les dates écrites et que **toutes les variables partagent la même dim `date`** (le bug qui avait figé `median_ratio`/`tail_ratio` à 4 alors que `exceedance_prob` montait à 24).

In [ ]:
import numpy as np
import xarray as xr

exc = xr.open_zarr(cfg.outputs.exceedance_store_uri, consolidated=False,
                   storage_options=STORAGE_OPTIONS)
dates = [str(d)[:10] for d in np.sort(exc["date"].values)]
print(f"Store exceedance : {len(dates)} dates  {dates[0]}..{dates[-1]}")
print("Variables        :", list(exc.data_vars))
nd = exc.sizes["date"]
desync = {v: int(exc[v].sizes.get("date", 0)) for v in exc.data_vars
          if exc[v].sizes.get("date") != nd}
if desync:
    print("DESYNC date détecté:", desync, "-> store corrompu (cf. bug writer _build_dataset).")
else:
    print(f"OK — toutes les variables alignées sur date={nd}.")

## 5. C3 — en local

C3 (risk CRMA) ne tourne **pas** ici : lancez-le sur votre machine quand C2 a écrit la fenêtre voulue. Il lit `exceedance-zarr` depuis MinIO.

```bash
# depuis le repo, en local (git bash) :
bash scripts/run_c3_local.sh 2024-09-01 2025-01-31
```

Le script lit les creds MinIO depuis `.env`, exécute `gik-icechain risk` sur la fenêtre, et écrit les GeoJSON dans `results/admin1_risk/`.
